In [14]:
import torch
from torch import nn
import numpy as np

torch.manual_seed(42)
np.random.seed(42) 

X = torch.tensor([[0., 0.],  #Example 1 
                  [0., 1.],  #Example 2
                  [1., 0.],  #Example 3
                  [1., 1.]]) #Example 4
    #Batch size = 4
    #Each Example has 2 Featuers

y = torch.tensor([[0.],  #Target for Example 1
                  [1.],  #Target for Example 2
                  [1.],  #Target for Example 3
                  [0.]]) #Target for Example 4
    

print("X shape:", X.shape)  # [4, 2]
print("y shape:", y.shape)  # [4, 1]

X shape: torch.Size([4, 2])
y shape: torch.Size([4, 1])


In [15]:
X_np = X.numpy()
y_np = y.numpy()

W1 = np.random.randn(2, 8) * 0.1  # 2 inputs -> 8 hidden neuron
b1 = np.zeros(8)                  # 8 biases (one per hidden neuron)
W2 = np.random.randn(8, 1) * 0.1  # 8 hidden -> 1 out put
b2 = np.zeros(1)                  # 1 bias (one per output neuron)

Z1 = X_np @ W1 + b1
assert Z1.shape == (4, 8)
                                # Expected: 4 examples, 8 hidden units

A1 = np.maximum(0, Z1)          # ReLU
assert A1.shape == (4, 8)
                                # Expected: same shape as Z1

logits = A1 @ W2 + b2
assert logits.shape == (4, 1)
                                # Expected: 4 examples, 1 logit each

probs = 1 / (1 + np.exp(-logits))
print(probs)

[[0.5       ]
 [0.50131282]
 [0.48942921]
 [0.49539531]]


In [16]:
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(2000):
    model.train()
    optimizer.zero_grad()
    logits = model(X)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch:4d}, loss={loss.item():.4f}")

Epoch    0, loss=0.7220
Epoch  200, loss=0.0337
Epoch  400, loss=0.0049
Epoch  600, loss=0.0020
Epoch  800, loss=0.0011
Epoch 1000, loss=0.0007
Epoch 1200, loss=0.0005
Epoch 1400, loss=0.0004
Epoch 1600, loss=0.0003
Epoch 1800, loss=0.0002


In [17]:
model.eval()
with torch.no_grad():
    logits = model(X)
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).float()

for i in range(4):
    print(
        "input:", X[i].tolist(),
        "target:", y[i].item(),
        "prob:", round(probs[i].item(), 4),
        "pred:", int(preds[i].item())
    )

input: [0.0, 0.0] target: 0.0 prob: 0.0001 pred: 0
input: [0.0, 1.0] target: 1.0 prob: 0.9999 pred: 1
input: [1.0, 0.0] target: 1.0 prob: 0.9998 pred: 1
input: [1.0, 1.0] target: 0.0 prob: 0.0002 pred: 0


In [18]:
nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

Sequential(
  (0): Linear(in_features=2, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

In [19]:
nn.Sequential(
    nn.Linear(2, 8),
    nn.Tanh(),
    nn.Linear(8, 1)
)

Sequential(
  (0): Linear(in_features=2, out_features=8, bias=True)
  (1): Tanh()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

In [20]:
nn.Sequential(
    nn.Linear(2, 8),
    nn.Linear(8, 1)
)

Sequential(
  (0): Linear(in_features=2, out_features=8, bias=True)
  (1): Linear(in_features=8, out_features=1, bias=True)
)

In [21]:
model_bad = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)   # wrong: input should be 8, not 3
)

In [22]:
model_bad(X)

tensor([[0.3199],
        [0.3824],
        [0.2443],
        [0.1500]], grad_fn=<AddmmBackward0>)

In [23]:
if torch.cuda.is_available():
    model = model.to("cuda")
    X = X.to("cuda")
    y = y.to("cuda")
    logits = model(X)

### Debugging Challenge Evidence
**The Error:** I deliberately created a shape mismatch by setting the second linear to 'nn.Linear(3, 1)'. Running 'model_bad(x)' produced:
'RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x3 and 3x1)'
**The Fix:** I changed the second layer to 'nn.Linear(8, 1)' to match the 8 incoming features from the hidden layer. The model now runs successfully and outputs raw logits of shape '[4, 1]'.
**Device Check:** I added a check for 'torch.cuda.is_available()'. Since this enviroment is CPU-only, the block was skiped (blank output),  confirming the model is staying on the CPU.